In [1]:
from f5_tts.model.cfm import CFM

from f5_tts.model.backbones.unett import UNetT
from f5_tts.model.backbones.dit import DiT
from f5_tts.model.backbones.mmdit import MMDiT

from f5_tts.model.trainer import Trainer


import os
import sys

# sys.path.append(f"../../{os.path.dirname(os.path.abspath(__file__))}/third_party/BigVGAN/")

import hashlib
import re
import tempfile
from importlib.resources import files

import matplotlib

matplotlib.use("Agg")

import matplotlib.pylab as plt
import numpy as np
import torch
import torchaudio
import tqdm
from pydub import AudioSegment, silence
from transformers import pipeline
from vocos import Vocos

# from f5_tts.model import CFM
from num2words import num2words
import soundfile as sf
# import gradio as gr

2025-06-14 18:46:27.898171: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749926787.948573     329 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749926787.962846     329 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749926788.069263     329 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749926788.069291     329 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749926788.069293     329 computation_placer.cc:177] computation placer alr

In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)
# -----------------------------------------

target_sample_rate = 24000
n_mel_channels = 100
hop_length = 256
win_length = 1024
n_fft = 1024
mel_spec_type = "vocos"
# mel_spec_type = "bigvgan"
target_rms = 0.1
cross_fade_duration = 0.15
ode_method = "euler"
nfe_step = 32  # 16, 32
cfg_strength = 2.0
sway_sampling_coef = -1.0
speed = 1.0
fix_duration = None

# -----------------------------------------

_ref_audio_cache = {}
# load asr pipeline
asr_pipe = None

cuda


In [3]:
#UTILS_INFER

# load vocoder
def load_vocoder(is_local=False, local_path="", device=device):
    if mel_spec_type == "vocos":
        if is_local:
            print(f"Load vocos from local path {local_path}")
            vocoder = Vocos.from_hparams(f"{local_path}/config.yaml")
            state_dict = torch.load(f"{local_path}/pytorch_model.bin", map_location="cpu")
            vocoder.load_state_dict(state_dict)
            vocoder = vocoder.eval().to(device)
        else:
            print("Download Vocos from huggingface charactr/vocos-mel-24khz")
            vocoder = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device)
    elif mel_spec_type == "bigvgan":
        try:
            # from third_party.BigVGAN import bigvgan
            import bigvgan
        except ImportError:
            print("You need to follow the README to init submodule and change the BigVGAN source code.")
        if is_local:
            """download from https://huggingface.co/nvidia/bigvgan_v2_24khz_100band_256x/tree/main"""
            vocoder = bigvgan.BigVGAN.from_pretrained(local_path, use_cuda_kernel=False)
        else:
            vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_24khz_100band_256x", use_cuda_kernel=False)

        vocoder.remove_weight_norm()
        vocoder = vocoder.eval().to(device)
    return vocoder



def initialize_asr_pipeline(device=device, dtype=None):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    global asr_pipe
    asr_pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-large-v3-turbo",
        torch_dtype=dtype,
        device=device,
    )


# load model checkpoint for inference


def load_checkpoint(model, ckpt_path, device, dtype=None, use_ema=True):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    model = model.to(dtype)

    ckpt_type = ckpt_path.split(".")[-1]
    if ckpt_type == "safetensors":
        from safetensors.torch import load_file

        checkpoint = load_file(ckpt_path)
    else:
        checkpoint = torch.load(ckpt_path, weights_only=True)

    if use_ema:
        if ckpt_type == "safetensors":
            checkpoint = {"ema_model_state_dict": checkpoint}
        checkpoint["model_state_dict"] = {
            k.replace("ema_model.", ""): v
            for k, v in checkpoint["ema_model_state_dict"].items()
            if k not in ["initted", "step"]
        }

        # patch for backward compatibility, 305e3ea
        for key in ["mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"]:
            if key in checkpoint["model_state_dict"]:
                del checkpoint["model_state_dict"][key]

        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        if ckpt_type == "safetensors":
            checkpoint = {"model_state_dict": checkpoint}
        model.load_state_dict(checkpoint["model_state_dict"])

    return model.to(device)


# load model for inference



def remove_silence_edges(audio, silence_threshold=-42):
    # Remove silence from the start
    non_silent_start_idx = silence.detect_leading_silence(audio, silence_threshold=silence_threshold)
    audio = audio[non_silent_start_idx:]

    # Remove silence from the end
    non_silent_end_duration = audio.duration_seconds
    for ms in reversed(audio):
        if ms.dBFS > silence_threshold:
            break
        non_silent_end_duration -= 0.001
    trimmed_audio = audio[: int(non_silent_end_duration * 1000)]

    return trimmed_audio



# infer process: chunk text -> infer batches [i.e. infer_batch_process()]

# remove silence from generated wav


def remove_silence_for_generated_wav(filename):
    aseg = AudioSegment.from_file(filename)
    non_silent_segs = silence.split_on_silence(
        aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=500, seek_step=10
    )
    non_silent_wave = AudioSegment.silent(duration=0)
    for non_silent_seg in non_silent_segs:
        non_silent_wave += non_silent_seg
    aseg = non_silent_wave
    aseg.export(filename, format="wav")


# save spectrogram


def save_spectrogram(spectrogram, path):
    plt.figure(figsize=(12, 4))
    plt.imshow(spectrogram, origin="lower", aspect="auto")
    plt.colorbar()
    plt.savefig(path)
    plt.close()



In [4]:
#UTILS

import os
import random
from collections import defaultdict
from importlib.resources import files

import torch
from torch.nn.utils.rnn import pad_sequence

import jieba
from pypinyin import lazy_pinyin, Style


# seed everything
def seed_everything(seed=0):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# helpers


def exists(v):
    return v is not None


def default(v, d):
    return v if exists(v) else d


def traducir_numero_a_texto(texto):
    texto_separado = re.sub(r'([A-Za-z])(\d)', r'\1 \2', texto)
    texto_separado = re.sub(r'(\d)([A-Za-z])', r'\1 \2', texto_separado)
    
    def reemplazar_numero(match):
        numero = match.group()
        return num2words(int(numero), lang='es')

    texto_traducido = re.sub(r'\b\d+\b', reemplazar_numero, texto_separado)

    return texto_traducido


# convert char to pinyin
def convert_char_to_pinyin(text_list, polyphone=True):
    final_text_list = []
    god_knows_why_en_testset_contains_zh_quote = str.maketrans(
        {"“": '"', "”": '"', "‘": "'", "’": "'"}
    )  # in case librispeech (orig no-pc) test-clean
    custom_trans = str.maketrans({";": ","})  # add custom trans here, to address oov
    for text in text_list:
        char_list = []
        text = text.translate(god_knows_why_en_testset_contains_zh_quote)
        text = text.translate(custom_trans)
        for seg in jieba.cut(text):
            seg_byte_len = len(bytes(seg, "UTF-8"))
            if seg_byte_len == len(seg):  # if pure alphabets and symbols
                if char_list and seg_byte_len > 1 and char_list[-1] not in " :'\"":
                    char_list.append(" ")
                char_list.extend(seg)
            elif polyphone and seg_byte_len == 3 * len(seg):  # if pure chinese characters
                seg = lazy_pinyin(seg, style=Style.TONE3, tone_sandhi=True)
                for c in seg:
                    if c not in "。，、；：？！《》【】—...":
                        char_list.append(" ")
                    char_list.append(c)
            else:  # if mixed chinese characters, alphabets and symbols
                for c in seg:
                    if ord(c) < 256:
                        char_list.extend(c)
                    else:
                        if c not in "。，、；：？！《》【】—...":
                            char_list.append(" ")
                            char_list.extend(lazy_pinyin(c, style=Style.TONE3, tone_sandhi=True))
                        else:  # if is zh punc
                            char_list.append(c)
        final_text_list.append(char_list)

    return final_text_list


# filter func for dirty data with many repetitions


def repetition_found(text, length=2, tolerance=10):
    pattern_count = defaultdict(int)
    for i in range(len(text) - length + 1):
        pattern = text[i : i + length]
        pattern_count[pattern] += 1
    for pattern, count in pattern_count.items():
        if count > tolerance:
            return True
    return False



In [5]:
def get_tokenizer(dataset_name, tokenizer: str = "pinyin"):
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                - "char" for char-wise tokenizer, need .txt vocab_file
                - "byte" for utf-8 tokenizer
                - "custom" if you're directly passing in a path to the vocab.txt you want to use
    vocab_size  - if use "pinyin", all available pinyin types, common alphabets (also those with accent) and symbols
                - if use "char", derived from unfiltered character & symbol counts of custom dataset
                - if use "byte", set to 256 (unicode byte range)
    """
    if tokenizer in ["pinyin", "char"]:
        tokenizer_path = os.path.join(files("f5_tts").joinpath("../../data"), f"{dataset_name}_{tokenizer}/vocab.txt")
        # tokenizer_path ="./F5TTS/vocab_pinyin.txt"
        # tokenizer_path ="/home/jupyter/F5TTS/vocab_pinyin.txt"
        with open(tokenizer_path, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)
        assert vocab_char_map[" "] == 0, "make sure space is of idx 0 in vocab.txt, cuz 0 is used for unknown char"

    elif tokenizer == "byte":
        vocab_char_map = None
        vocab_size = 256

    elif tokenizer == "custom":
        with open(dataset_name, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)

    return vocab_char_map, vocab_size


In [6]:
def preprocess_ref_audio_text(ref_audio_orig, ref_text, clip_short=False, show_info=print, device=device):
    show_info("Converting audio...")
    print("Converting audio...")
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        aseg = AudioSegment.from_file(ref_audio_orig)

        if clip_short:
            # 1. try to find long silence for clipping
            non_silent_segs = silence.split_on_silence(
                aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=1000, seek_step=10 
                # aseg, min_silence_len=2000, silence_thresh=-50, keep_silence=1000, seek_step=10 
            )
            non_silent_wave = AudioSegment.silent(duration=0)
            for non_silent_seg in non_silent_segs:
                if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                    show_info("Audio is over 15s, clipping short. (1)")
                    break
                non_silent_wave += non_silent_seg

            # 2. try to find short silence for clipping if 1. failed
            if len(non_silent_wave) > 15000:
                non_silent_segs = silence.split_on_silence(
                    aseg, min_silence_len=100, silence_thresh=-40, keep_silence=1000, seek_step=10
                )
                non_silent_wave = AudioSegment.silent(duration=0)
                for non_silent_seg in non_silent_segs:
                    if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                        show_info("Audio is over 15s, clipping short. (2)")
                        break
                    non_silent_wave += non_silent_seg

            aseg = non_silent_wave

            # 3. if no proper silence found for clipping
            if len(aseg) > 15000:
                aseg = aseg[:15000]
                show_info("Audio is over 15s, clipping short. (3)")

        aseg = remove_silence_edges(aseg) + AudioSegment.silent(duration=50)
        aseg.export(f.name, format="wav")
        ref_audio = f.name

    # Compute a hash of the reference audio file
    with open(ref_audio, "rb") as audio_file:
        audio_data = audio_file.read()
        audio_hash = hashlib.md5(audio_data).hexdigest()

    global _ref_audio_cache
    if audio_hash in _ref_audio_cache:
        # Use cached reference text
        show_info("Using cached reference text...")
        ref_text = _ref_audio_cache[audio_hash]
    else:
        print("ref_text ",len(ref_text),ref_text)
        if not ref_text.strip():
        # if len(ref_text)==0:
        # if ref_text =="":
            
            global asr_pipe
            if asr_pipe is None:
                initialize_asr_pipeline(device=device)
            show_info("No reference text provided, transcribing reference audio...")
            ref_text = asr_pipe(
                ref_audio,
                chunk_length_s=30,
                batch_size=128,
                generate_kwargs={"task": "transcribe"},
                return_timestamps=False,
            )["text"].strip()
            
            show_info("Finished transcription")
        else:
            show_info("Using custom reference text...")
        # Cache the transcribed text
        _ref_audio_cache[audio_hash] = ref_text
    print("ref_text: ",ref_text)
    # Ensure ref_text ends with a proper sentence-ending punctuation
    if not ref_text.endswith(". ") and not ref_text.endswith("。"):
        if ref_text.endswith("."):
            ref_text += " "
        else:
            ref_text += ". "

    return ref_audio, ref_text


In [7]:
def load_model(
    model_cls,
    model_cfg,
    ckpt_path,
    mel_spec_type=mel_spec_type,
    vocab_file="",
    ode_method=ode_method,
    use_ema=True,
    device=device,
):
    if vocab_file == "":
        # vocab_file = str(files("f5_tts").joinpath("infer/examples/vocab.txt"))
        vocab_file = "./F5TTS/vocab.txt"
    tokenizer = "custom"
    # tokenizer = "pinyin"
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                    - "char" for char-wise tokenizer, need .txt vocab_file
                    - "byte" for utf-8 tokenizer
                    - "custom" if you're directly passing in a path to the vocab.txt you want to use
    """

    print("\nvocab : ", vocab_file)
    print("tokenizer : ", tokenizer)
    print("model : ", ckpt_path, "\n")

    vocab_char_map, vocab_size = get_tokenizer(vocab_file, tokenizer)
    model = CFM(
        transformer=model_cls(**model_cfg, text_num_embeds=vocab_size, mel_dim=n_mel_channels),
        mel_spec_kwargs=dict(
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mel_channels=n_mel_channels,
            target_sample_rate=target_sample_rate,
            mel_spec_type=mel_spec_type,
        ),
        odeint_kwargs=dict(
            method=ode_method,
        ),
        vocab_char_map=vocab_char_map,
    ).to(device)

    dtype = torch.float32 if mel_spec_type == "bigvgan" else None
    model = load_checkpoint(model, ckpt_path, device, dtype=dtype, use_ema=use_ema)

    return model



In [8]:
ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozRealSeriaMarcos3.wav"))+"/VozRealSeriaMarcos3.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozPresentadorCirco.wav"))+"/VozPresentadorCirco.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/LuciferCuandoCortasModif.wav"))+"/LuciferCuandoCortasModif.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozPresentadorMarcos3.wav"))+"/VozPresentadorMarcos3.wav"#"./F5TTS/FL.wav", #Ruta al audio


ref_text_input=""
# ref_text_input='Laura medía un metro setenta, tenía los pies palmeados de nacimiento y marcas de nacimiento iguales en ambos muslos. Una tenía la forma de su padre y la otra la de su madre, o eso decía ella. A mí me parecían salpicaduras negras más o menos iguales, la izquierda ligeramente más grande, más dentada que la otra, ambas moteadas por manchas de marrón oscuro. Un solo pelo salía largo de la más suave. Me las enseñó tres semanas después de conocernos en un banco mojado de un parque a las tres de la madrugada. Sus pies palmeados aparecieron primero, pero no le preocupaban mucho. Dijo que no veía el alboroto. No desde el instituto. Sus curvas y su baja estatura la convertían en una pésima nadadora y las otras chicas habían hecho un deporte de señalar la ironía, entre otras cosas más mezquinas. Laura era irónica, en muchos aspectos más que eso. La marca de su padre era la más dolorosa. Se le llenaban los ojos de lágrimas mientras trazaba los contornos del borde más afilado y explicaba el significado de su extraña geometría. Pero era difícil seguir después de la parte del bastón de madera. Hablaba a trompicones y cada tres o cuatro palabras sonaban a árabe; y resultó que era árabe. El árabe es un idioma impresionante. Hasta las indicaciones para ir al baño suenan poéticas en árabe. Al menos para mí.   Nunca le pregunté qué significaba, no me preguntes por qué, y traducir sus palabras ahora, después de lo que pasó -después de lo que hizo- es lo más alejado de cualquier cosa que pueda imaginarme haciendo por elección propia.   Lo mismo ocurrió con la marca por parte de madre, pero fue el italiano el idioma al que se dirigió entonces.  Estaba demasiado hipnotizado para decir nada.  Sólo seguía sus expresiones e inflexiones lo mejor que podía. Cuando su lengua cambió, sentí un dolor diferente, más intenso, por lo que pude ver, en la zona donde crecía el pelo largo.  Era casi imposible no abrazarla cuando se estremecía. Y entonces el propio pelo me hizo sonreír, lo suficiente como para mostrar lo compleja que era aquella relación.  Nunca había conocido a una chica con metáforas naturales en las piernas. No es que ella lo viera así, claro.   No podía ser más entrañable, y su trauma hizo que se me encendieran las entrañas. No creo que importara mucho que yo no lo siguiera todo.  Todo giraba en torno a ella. Yo era su seguridad más bien, que era como había sido desde el principio.  Desde la noche en que volvía tarde a casa y la encontré agarrada al otro lado de la barrera. La del puente alto sobre el río.'


remove_silence=False #El modelo tiende a producir silencios, especialmente en audios más largos. Podemos eliminar manualmente los silencios si es necesario. Ten en cuenta que esta es una característica experimental y puede producir resultados extraños. Esto también aumentará el tiempo de generación.
cross_fade_duration_slider=0.15 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
# cross_fade_duration_slider=1.0 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
speed_slider=2.0#Ajusta la velocidad del audio. Entre 0.3 y 2.0


In [9]:
# def infer(ref_audio_orig, ref_text, gen_text, remove_silence, cross_fade_duration=0.15, speed=1):
# ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=False)
ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=True)
# print(ref_text)


Converting audio...
Converting audio...
ref_text  0 


Device set to use cuda
/usr/local/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


No reference text provided, transcribing reference audio...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Finished transcription
ref_text:  Al pasar de 10.000 a 5.000 kilómetros, las lecturas de radiación se dispararon y los niveles de microgravedad comenzaron a fluctuar sin motivo aparente. Ante ellos se erguía la silueta de aquello que el espacio había ocultado durante milenios.


In [47]:
# Susurros de umbraluna
# Parte 1
gen_text_input='Hay cosas en este mundo que la medicina no puede explicar. Cosas que te hacen cuestionar todo lo que creías saber sobre la vida... y sobre la muerte. La siguiente historia viene de un manuscrito encontrado entre los papeles personales del Dr. Aurelius Mendoza, tras su muerte en 1875. Los registros históricos confirman su trabajo en el pueblo de Umbraluna entre 1844 y 1847, así como su posterior práctica especializada en trastornos mentales en la capital. Aunque la veracidad de los eventos sobrenaturales descritos no puede ser verificada, el documento ofrece una fascinante perspectiva sobre las creencias y prácticas médicas del siglo XIX. Mi nombre es Aurelius Mendoza, y en 1844 era un médico recién graduado de la Universidad de medicina en la capital. Tenía el cabello prematuramente canoso —una característica que heredé de mi padre— y una cicatriz que atravesaba mi ceja izquierda, recuerdo de una caída durante mi infancia. Era un hombre de ciencia, escéptico por naturaleza, que creía únicamente en lo que podía ver, tocar y explicar mediante la razón. Fue por una carta de mi tío Sebastián que llegué a Umbraluna. El viejo doctor que había servido al pueblo durante décadas había fallecido súbitamente, y necesitaban urgentemente un reemplazo. El pueblo se encontraba en las montañas del norte, un lugar donde las brumas persistían incluso en los días más soleados y donde los antiguos robles parecían susurrar secretos olvidados. Al llegar, me instalé en la vivienda que había pertenecido al doctor anterior, una casa de piedra de dos pisos ubicada en el centro del pueblo. Lo primero que me llamó la atención fue la biblioteca del difunto médico: estantes repletos de libros de medicina, pero también volúmenes extraños sobre hierbas místicas, rituales antiguos y fenómenos inexplicables. Como hombre racional, descarté estas lecturas como supersticiones de un anciano excéntrico. Los primeros meses transcurrieron sin incidentes. Atendía los partos, curaba fiebres, vendaba heridas y poco a poco ganaba la confianza de los habitantes. Sin embargo, había algo en el aire de Umbraluna que me inquietaba: una sensación constante de ser observado, como si ojos invisibles siguieran cada uno de mis movimientos. Fue en octubre cuando conocí a Esperanza Valdivia, la comadrona del pueblo. Era una mujer de unos cincuenta años, de cabello completamente blanco recogido siempre en un moño perfecto, y ojos verdes tan penetrantes que parecían leer el alma. Tenía la peculiaridad de usar siempre un collar de cuentas de ámbar que, según decía, había pertenecido a su abuela. Esperanza era respetada y temida a partes iguales, pues se rumoreaba que poseía conocimientos que iban más allá de la medicina tradicional. —Doctor Mendoza —me dijo la primera vez que coincidimos en un parto difícil—, este pueblo guarda secretos que es mejor no despertar. El doctor anterior lo sabía, por eso vivió tantos años aquí sin problemas. Sus palabras me parecieron las típicas advertencias supersticiosas de la gente rural, pero algo en su tono me hizo estremecer. Quien también despertó mi curiosidad fue Próspero Castell, el boticario. Era un hombre delgado como una vara, de unos cuarenta años, con una barba rala y rojiza que contrastaba con su palidez extrema. Lo que más me impresionaba de él eran sus manos: increíblemente largas y huesudas, con dedos que parecían arañas cuando manipulaba sus frascos y pociones. Próspero había heredado la botica de su padre y conocía cada hierba, cada remedio tradicional del lugar. Siempre vestía un delantal manchado de verde por las hierbas que preparaba constantemente. El primer incidente sobrenatural ocurrió una noche de noviembre. Había sido llamado para atender a Remedios Herrera, una anciana de ochenta años que todos conocían por su cabello completamente blanco y su costumbre de hablar sola por las calles. La llamaban "la loca Remedios", pero yo había descubierto que su supuesta locura se debía a una demencia senil completamente explicable. Esa noche, Remedios estaba en su lecho de muerte. Su respiración era laboriosa y sus ojos, normalmente confusos, brillaban con una claridad perturbadora. Cuando me acerqué para examinarla, me agarró la muñeca con una fuerza sorprendente para su estado. —Doctor —susurró con una voz que parecía venir de ultratumba—, ellos saben que usted está aquí. Los que murieron sin descanso. Vienen por las noches... primero susurran, luego tocan, después... después vienen por los vivos. Sus palabras me helaron la sangre, pero mantuve mi compostura profesional. Le administré láudano para calmar su agitación y me quedé a velarla. Fue entonces cuando comencé a escuchar los susurros. Al principio pensé que eran alucinaciones auditivas, producto del cansancio y la tensión. Pero los sonidos eran claros: voces ininteligibles que parecían provenir de las paredes mismas de la casa. Remedios había muerto silenciosamente alrededor de la medianoche, pero los susurros continuaron hasta el amanecer. A partir de esa noche, los fenómenos se intensificaron. Cada vez que alguien moría en el pueblo —algo que ocurría con más frecuencia de lo normal—, yo experimentaba manifestaciones inexplicables. Objetos que se movían solos en mi consulta, sombras que se desplazaban por los rincones de mi visión, y siempre, siempre, esos susurros que parecían llamarme por mi nombre. Decidí consultar con Esperanza, quien me recibió en su pequeña casa llena de hierbas secas colgando del techo. El aroma era abrumador: lavanda, romero, salvia y otras plantas que no pude identificar. —Los espíritus inquietos han despertado, doctor —me explicó mientras preparaba un té de hierbas—. El doctor anterior, que en paz descanse, conocía los rituales para mantenerlos en calma. Usted, sin saberlo, ha roto el equilibrio. —Esperanza, soy un hombre de ciencia. No puedo creer en fantasmas y espíritus. Ella me miró con una sonrisa triste. —La ciencia no lo abarca todo, doctor. Hay fuerzas en este mundo que la razón no puede explicar. Su predecesor lo aprendió por las malas. Me contó entonces que el doctor anterior había llegado al pueblo con la misma actitud escéptica que yo. Pero después de varios encuentros con lo inexplicable, había terminado por estudiar los antiguos conocimientos del lugar, aprendiendo a coexistir con las fuerzas sobrenaturales que habitaban Umbraluna. —¿Qué clase de fuerzas? —pregunté, aunque parte de mí ya no quería conocer la respuesta. —Almas en pena, doctor. Personas que murieron con asuntos pendientes, con dolor no resuelto. Este pueblo es muy antiguo, y muchas tragedias han ocurrido aquí. Esas almas necesitan ser... apaciguadas. Esperanza me enseñó entonces algunos de los libros que habían pertenecido al doctor anterior, textos que yo había ignorado por considerarlos supersticiones. En ellos encontré descripciones detalladas de los fenómenos que estaba experimentando, así como métodos para lidiar con ellos. Pero mi naturaleza científica se revelaba contra estas explicaciones. Decidí buscar una segunda opinión en Próspero, el boticario. Lo encontré en su tienda, preparando una mezcla que despedía un olor dulzón y nauseabundo. Sus dedos largos y huesudos manipulaban los ingredientes con precisión quirúrgica. —Ah, doctor Mendoza —me saludó sin levantar la vista—. Supongo que viene por los visitantes nocturnos. —¿Visitantes nocturnos? Próspero me miró entonces con sus ojos pequeños y brillantes. —Los muertos, doctor. Los que no pueden descansar. Su llegada los ha agitado. El doctor anterior había logrado mantener un... digamos, un acuerdo con ellos. Pero ahora todo ha cambiado. Me explicó que en Umbraluna existía una antigua tradición: cada doctor que servía al pueblo debía realizar ciertos rituales para mantener en paz a los espíritus de aquellos que habían muerto con dolor. El doctor anterior había cumplido fielmente con estas obligaciones durante décadas. —¿Qué tipo de rituales? —pregunté, sintiendo que mi mundo racional se desmoronaba. —Ofrendas, oraciones específicas, y sobre todo, escuchar sus historias. Los muertos necesitan ser recordados, necesitan que alguien conozca su dolor para poder descansar. Esa misma noche, decidí poner a prueba esta teoría. Si realmente había espíritus en mi casa, intentaría comunicarme con ellos de manera racional y científica. Preparé papel y pluma para tomar notas detalladas de cualquier fenómeno que ocurriera. No tuve que esperar mucho. Cerca de la medianoche, los susurros comenzaron de nuevo, pero esta vez parecían más urgentes, más claros. Me armé de valor y hablé en voz alta: —Si hay alguien aquí, si realmente existen, necesito entender qué quieren de mí. El silencio que siguió fue ensordecedor. Luego, gradualmente, comencé a percibir una presencia en la habitación. El aire se volvió frío y espeso, y una sensación de profunda tristeza me invadió, como si las emociones de otra persona se hubieran superpuesto a las mías. Fue entonces cuando la vi por primera vez: una figura translúcida de mujer joven, vestida con ropas del siglo pasado. Su rostro estaba marcado por una expresión de dolor infinito, y sus ojos me miraban con súplica desesperada. —Ayúdame —susurró con una voz que parecía venir de muy lejos—. Mi niño... mi pequeño Tomás... La aparición duró apenas unos segundos, pero fue suficiente para cambiar para siempre mi percepción de la realidad. Al día siguiente, busqué información sobre esta mujer en los registros del pueblo. En los archivos de la pequeña iglesia, encontré la historia de Concepción Morales, una joven madre que había muerto en 1798 durante un incendio que también cobró la vida de su hijo pequeño, Tomás. Según los registros, Concepción había intentado desesperadamente salvar a su niño, pero ambos perecieron en las llamas. Esa noche regresé a mi casa con un nuevo propósito. Si esta mujer necesitaba ayuda para descansar en paz, yo intentaría dársela. Cuando volvió a aparecer, le hablé con gentileza: —Concepción, sé quién eres. Sé lo que pasó con tu hijo Tomás. No fue tu culpa. Hiciste todo lo que pudiste. La figura se hizo más sólida, y por primera vez, vi una expresión de alivio en su rostro. —¿De verdad lo cree, doctor? ¿No fui una mala madre? —Fuiste una madre valiente que dio su vida intentando salvar a su hijo. Tomás está contigo ahora, en paz. Es hora de que ambos descansen. La aparición me sonrió entonces, una sonrisa de gratitud infinita, y se desvaneció lentamente. Esa noche dormí sin susurros por primera vez en semanas. Pero Concepción era solo la primera de muchas almas inquietas. Durante los siguientes meses, aprendí que mi papel como médico del pueblo iba más allá de curar a los vivos: también debía ayudar a sanar las heridas emocionales de los muertos. Cada espíritu que se me aparecía tenía una historia diferente. Estaba Patricio Velázquez, un minero que había muerto en un derrumbe en 1823, atormentado por la culpa de haber dejado huérfanos a sus tres hijos. Había muerto pensando que su familia moriría de hambre sin él, sin saber que sus hermanos se habían hecho cargo de los niños, quienes crecieron sanos y prósperos. También conocí a Valentina Torres, una joven que había muerto de tuberculosis en 1830, convencida de que había contagiado la enfermedad a su prometido. En realidad, él había vivido una larga vida y había muerto de vejez años después, recordándola siempre con amor. Cada historia era diferente, pero todas compartían un elemento común: los espíritus estaban atrapados por emociones no resueltas, por culpas infundadas o por el dolor de haber dejado asuntos inconclusos. Mi labor se convirtió en investigar sus historias, encontrar la verdad y ayudarlos a encontrar paz. Esperanza se convirtió en mi guía en este extraño nuevo aspecto de mi profesión. Me enseñó rituales de purificación, oraciones protectoras y, sobre todo, la importancia de escuchar con compasión. —Los muertos necesitan ser comprendidos tanto como los vivos, doctor —me decía mientras preparamos ofrendas de flores y velas—. Su dolor es real, aunque su forma de existir sea diferente. Próspero también contribuyó con su conocimiento, preparando inciensos especiales y mezclas de hierbas que, según él, facilitaban la comunicación con el más allá. —No se trata de magia, doctor —me explicaba mientras molía raíces aromáticas—. Es como... como preparar el ambiente adecuado para una conversación difícil. Estas hierbas calman tanto a los vivos como a los muertos. El caso más desafiante llegó en la primavera de 1845. Se trataba de un espíritu diferente a todos los anteriores: violento, lleno de ira, que hacía temblar las paredes de mi casa y llenaba mis noches de pesadillas terribles. La primera vez que se manifestó, sentí como si el aire mismo se volviera tóxico. Una presencia maligna llenó mi habitación, y una voz ronca y amenazante gritó: —¡Fuera de aquí! ¡Esta es mi casa! ¡Mi lugar! Los objetos comenzaron a volar por la habitación, y sentí manos invisibles que intentaban estrangularme. Logré escapar de la casa, pero sabía que tendría que enfrentar a este espíritu eventualmente. Consulté con Esperanza, quien palideció al escuchar mi descripción. —Es Bartolomé Núñez —susurró—. El doctor anterior nunca logró ayudarlo a encontrar paz. Era... era un hombre muy malvado en vida. La historia de Bartolomé Núñez era la más oscura del pueblo. Había sido un usurero despiadado que vivía en la casa que ahora era mía. Durante décadas había aprovechado la pobreza de los habitantes, cobrando intereses abusivos y arrebatando propiedades a familias enteras. Había muerto solo y odiado por todos en 1815. —¿Por qué el doctor anterior no pudo ayudarlo? —pregunté. —Porque Bartolomé no quiere ser ayudado —explicó Esperanza—. Algunos espíritus se aferran a su dolor y su ira porque es lo único que conocen. Prefieren el sufrimiento familiar a la incertidumbre de la paz. Durante semanas, Bartolomé convirtió mi vida en un infierno. Rompía instrumentos médicos, derramaba medicinas, y llenaba mis noches de visiones horrendas de sus víctimas. Más de una vez consideré abandonar Umbraluna, pero sabía que si huía, otro médico tendría que enfrentar el mismo problema. Fue Próspero quien me sugirió una estrategia diferente. —No trate de convencerlo de encontrar paz, doctor —me aconsejó mientras preparaba una mezcla especial de hierbas protectoras—. En lugar de eso, establezca límites. Dígale que puede quedarse, pero que debe respetar ciertas reglas. La confrontación final ocurrió en una noche de tormenta. Bartolomé se manifestó con más fuerza que nunca, haciendo temblar toda la casa. En lugar de huir, me planté en el centro de mi sala y le hablé con firmeza: —Bartolomé Núñez, puedes quedarte en esta casa si eso es lo que deseas, pero no interferirás con mi trabajo. La gente de este pueblo necesita atención médica, y no permitiré que tu ira les cause daño. El espíritu rugió de rabia, pero continué: —Tus víctimas ya no están aquí. Tu sufrimiento solo te daña a ti mismo. Puedes elegir la paz, o puedes elegir seguir sufriendo, pero no arrastrarás a otros contigo. Hubo un silencio tenso, y luego la presencia maligna comenzó a desvanecerse. Desde esa noche, aunque aún podía sentir la presencia de Bartolomé en la casa, nunca más interfirió con mi trabajo. Habíamos llegado a una especie de tregua. Los años siguientes transcurrieron en una extraña normalidad. Continué ejerciendo como médico, atendiendo tanto a vivos como a muertos. Los habitantes del pueblo gradualmente se acostumbraron a mi particular forma de practicar la medicina, y algunos incluso comenzaron a traerme historias de familiares fallecidos que necesitaban ayuda para encontrar paz. Desarrollé métodos sistemáticos para investigar las historias de los espíritus, creando archivos detallados de cada caso. Descubrí que la mayoría de las almas inquietas simplemente necesitaban que alguien conociera su verdad, que reconociera su sufrimiento y les asegurara que no habían sido olvidadas. Mi relación con Esperanza se profundizó durante este tiempo. Ella se convirtió no solo en mi colaboradora, sino en mi amiga más cercana. Una tarde, mientras compartíamos té en su casa, me hizo una confesión sorprendente: —Doctor, debo decirle algo. Yo también puedo ver a los espíritus. Siempre he podido hacerlo. Por eso sabía que usted era diferente desde el primer día. —¿Por qué no me lo dijiste antes? —Porque necesitaba que descubriera por sí mismo que estas cosas son reales. Si se lo hubiera dicho desde el principio, habría pensado que era solo una vieja supersticiosa. Esperanza me contó entonces que había heredado esta habilidad de su abuela, quien también había sido comadrona en el pueblo. Me explicó que algunas personas nacían con la capacidad de percibir el mundo espiritual, y que yo era una de ellas. —Por eso los espíritus se sienten atraídos hacia usted, doctor. Pueden sentir que es capaz de verlos y escucharlos realmente. Esta revelación cambió mi comprensión de todo lo que había vivido. No era solo que hubiera desarrollado esta habilidad por las circunstancias; había nacido con ella, y las experiencias en Umbraluna simplemente habían despertado un don que siempre había estado latente. Próspero también tenía secretos que compartir. Una noche, mientras me ayudaba a preparar ofrendas para un grupo de espíritus particularmente inquietos, me reveló que él también podía sentir presencias sobrenaturales, aunque no podía verlas tan claramente como yo. —Mi familia ha sido guardiana de los secretos de este lugar durante generaciones —me explicó—. Mi padre me enseñó las recetas tradicionales no solo para curar enfermedades del cuerpo, sino también para calmar las dolencias del alma. Me mostró entonces fórmulas secretas que habían sido pasadas de padre a hijo: inciensos para facilitar la comunicación con los muertos, ungüentos para protegerse de espíritus malignos, y pociones para purificar lugares con mucha actividad paranormal. —El doctor anterior era como usted —continuó Próspero—. Al principio escéptico, pero con el don natural. Por eso pudo aprender tan rápido y ser tan efectivo ayudando a las almas perdidas. Gradualmente, comencé a entender que mi llegada a Umbraluna no había sido una coincidencia. El pueblo había necesitado a alguien con mis habilidades específicas, y de alguna manera, el destino había conspirado para traerme allí. El momento más revelador llegó cuando finalmente decidí explorar completamente la biblioteca de mi predecesor. En un compartimento secreto detrás de los libros, encontré un diario personal que documentaba todas sus experiencias con lo sobrenatural. Las primeras páginas mostraban el mismo escepticismo que yo había sentido, seguido por la misma confusión y terror inicial. Pero las entradas posteriores revelaban un hombre que había encontrado propósito y satisfacción en su singular vocación. "Hoy ayudé a María Santos a encontrar paz", escribía en una entrada de 1840. "Había estado atormentada durante décadas por la culpa de haber sobrevivido a la epidemia que mató a sus hermanos. Le aseguré que no fue su culpa, que había hecho todo lo posible por cuidarlos. Su alivio fue palpable, y finalmente pudo descansar." Otra entrada decía: "He llegado a comprender que ser médico en Umbraluna significa algo más profundo que curar enfermedades físicas. Somos sanadores de heridas emocionales que trascienden la muerte. Es una responsabilidad sagrada." La última entrada del diario estaba fechada apenas una semana antes de su muerte: "Siento que mi tiempo aquí llega a su fin. He hecho todo lo que pude por las almas perdidas de este lugar, pero sé que vendrá alguien más, alguien joven y fuerte que continuará este trabajo. Los espíritus me han susurrado que ya está cerca." Leer estas palabras me provocó escalofríos. Mi predecesor había sabido de mi llegada antes de morir, y de alguna manera había preparado todo para mi transición a este extraño papel. Durante mis últimos años en Umbraluna, perfeccioné mis métodos y ayudé a docenas de espíritus a encontrar paz.'

In [74]:
# Susurros de umbraluna
# Parte 2
gen_text_input='Cada caso me enseñó algo nuevo sobre la naturaleza del alma humana y la importancia de la compasión, tanto para los vivos como para los muertos. Desarrollé rituales personales: encendía velas especiales que Próspero preparaba con cera de abejas y hierbas benditas, quemaba incienso de lavanda y romero para crear un ambiente de calma, y siempre comenzaba cada sesión con una oración de protección que Esperanza me había enseñado. La más emotiva de todas mis experiencias fue cuando finalmente logré ayudar a una familia completa de espíritus. Se trataba de los Herrera, que habían muerto en el mismo incendio que Concepción Morales en 1798. Durante décadas habían vagado juntos, pero separados por sus propias culpas individuales. El padre, Alejandro, se culpaba por no haber detectado el fuego a tiempo. La madre, Carmen, se atormentaba por no haber podido salvar a su hija menor. Los tres hijos —Esperanza, Miguel y la pequeña Ana— se sentían culpables por haber causado el incendio accidentalmente mientras jugaban con velas. Cuando todos se manifestaron simultáneamente en mi casa una noche de invierno, fue una experiencia abrumadora. La habitación se llenó de voces superpuestas, cada una relatando su versión de la tragedia, cada una asumiendo la culpa total. Les pedí silencio y les hablé como lo haría a una familia viva que estuviera atravesando una crisis: —Todos ustedes están aquí porque se aman. Murieron juntos porque trataron de protegerse unos a otros. No hubo villanos en esa tragedia, solo una familia que hizo todo lo que pudo en circunstancias terribles. Lentamente, uno por uno, logré que cada miembro de la familia perdonara no solo a los demás, sino también a sí mismo. Cuando finalmente se desvanecieron, fueron tomados de las manos, caminando juntos hacia una luz dorada que llenó toda la habitación. Esa noche experimenté una sensación de completud que nunca antes había sentido. Comprendí que mi verdadera vocación no era solo curar cuerpos enfermos, sino sanar almas heridas. Sin embargo, amigo mío, debo confesarte que todo este conocimiento y estas experiencias tenían un precio. Vivir constantemente en el límite entre el mundo de los vivos y el de los muertos comenzó a desgastarme física y emocionalmente. Desarrollé una sensibilidad extrema que me permitía percibir la presencia de espíritus incluso cuando no se manifestaban directamente. Podía sentir el dolor de los muertos como si fuera propio, y sus historias de sufrimiento comenzaron a pesarme en el alma. Esperanza notó estos cambios en mí y comenzó a preocuparse. —Doctor, debe tener cuidado —me advirtió una tarde mientras preparaba un té de hierbas calmantes—. Llevar tanto dolor ajeno puede consumirlo. Debe aprender a proteger su propia alma. Me enseñó técnicas de purificación espiritual: baños con sales especiales que Próspero preparaba, meditaciones protectoras, y rituales para limpiar mi energía después de cada encuentro con espíritus. —Recuerde —me decía— que usted está vivo. No debe perderse en el mundo de los muertos, por muy noble que sea su misión. A pesar de sus consejos y cuidados, comencé a experimentar efectos secundarios preocupantes. Mis cabellos, ya prematuramente canosos, se volvieron completamente blancos antes de los cuarenta años. Desarrollé una palidez que me hacía parecer fantasmal, y mis ojos adquirieron una expresión distante que asustaba a algunos pacientes. Más perturbador aún, comencé a tener visiones espontáneas de espíritus en otros lugares. Cuando visitaba pueblos vecinos por motivos médicos, podía ver almas inquietas que vagaban por las calles, y sus susurros se mezclaban con los sonidos de la vida cotidiana. Fue entonces cuando tomé la decisión más difícil de mi vida: dejar Umbraluna. No fue fácil. El pueblo se había convertido en mi hogar, y la gente me había aceptado completamente, incluyendo mis peculiares métodos. Esperanza y Próspero se habían convertido en mi familia elegida, y la idea de separarme de ellos me dolía profundamente. Pero sabía que si continuaba, terminaría perdiendo mi cordura o, peor aún, mi propia alma se quedaría atrapada entre mundos. Antes de partir, dediqué varios meses a entrenar a un joven sacerdote llamado Padre Luciano Morales, quien había mostrado sensibilidad hacia lo sobrenatural. Aunque no tenía el don completo que yo poseía, podía aprender los rituales básicos y mantener la paz que habíamos logrado establecer. —No todos pueden ver a los espíritus como usted, doctor —me dijo el Padre Luciano durante una de nuestras últimas sesiones de entrenamiento—, pero todos podemos escuchar con compasión y ofrecer consuelo. Le dejé todos los libros relevantes, mis notas detalladas sobre cada caso, y las fórmulas que Próspero me había enseñado. También le enseñé las oraciones protectoras y los rituales de purificación. Mi despedida de Esperanza fue especialmente dolorosa. Esta mujer extraordinaria había sido mi guía, mi maestra y mi amiga más querida. —Usted ha hecho más bien en estos pocos años que muchos en toda una vida —me dijo mientras me entregaba un último frasco de hierbas protectoras—. Los espíritus de Umbraluna estarán eternamente agradecidos. Próspero me obsequió una mezcla especial de hierbas que, según él, me protegerían donde quiera que fuera. —Lleve siempre un poco de Umbraluna con usted, doctor —me dijo con una sonrisa triste—. Y recuerde que siempre tendrá un hogar aquí. Cuando finalmente dejé el pueblo en la primavera de 1847, llevaba conmigo no solo mis pertenencias físicas, sino también un conocimiento profundo sobre la naturaleza del alma humana que cambiaría para siempre mi forma de ejercer la medicina. Ahora, tres años después, he establecido mi práctica en la capital, donde me he especializado en lo que los médicos modernos llaman "enfermedades mentales". Utilizando lo que aprendí en Umbraluna, he desarrollado métodos revolucionarios para tratar la melancolía, la histeria y otros males del alma. Aunque ya no veo espíritus regularmente, mi experiencia me ha dado una comprensión única del sufrimiento humano. Puedo detectar el dolor no resuelto en mis pacientes vivos con la misma claridad con que antes percibía las penas de los muertos. Mi cicatriz en la ceja izquierda se ha vuelto más pronunciada con los años, y mis cabellos completamente blancos me hacen parecer mayor de lo que soy, pero estos cambios físicos son un pequeño precio por la sabiduría que he adquirido. Aún mantengo correspondencia con Esperanza y Próspero. Me escriben regularmente para contarme cómo van las cosas en Umbraluna. El Padre Luciano ha demostrado ser un digno sucesor, aunque admite que nunca ha tenido una experiencia tan intensa como las que yo viví. Ocasionalmente, aún tengo encuentros con espíritus, pero ahora son más gentiles, más breves. Es como si mi tiempo en Umbraluna me hubiera marcado permanentemente, pero de una manera que ya no me consume. ¿Por qué te cuento todo esto, amigo mío? Porque creo que es importante que alguien más conozca esta historia. No busco que me creas —sé lo fantástico que suena todo esto—, pero si alguna vez te encuentras en circunstancias similares, si alguna vez percibes presencias inexplicables o escuchas susurros en la noche, quiero que sepas que no estás loco. Hay más misterios en este mundo de los que nuestra ciencia puede explicar actualmente. Eso no significa que debamos abandonar la razón, sino que debemos expandir nuestra comprensión de lo que es posible. Los muertos no descansan siempre en paz. Algunos necesitan ayuda para resolver los asuntos que los mantienen atados a este mundo. Y a veces, los vivos tenemos la capacidad y la responsabilidad de brindarles esa ayuda. He aprendido que la compasión es la medicina más poderosa que existe, tanto para los vivos como para los muertos. El sufrimiento trasciende la barrera de la muerte, pero también lo hace el amor y la comprensión. Cada noche, antes de dormir, enciendo una pequeña vela y susurro una oración por todas las almas que conocí en Umbraluna. No es por superstición, sino por gratitud. Ellos me enseñaron que ser médico significa algo más profundo que curar enfermedades: significa ser un sanador de almas. Hace pocas semanas recibí una carta de Esperanza que me llenó de una mezcla de nostalgia y preocupación. Me escribía para contarme que un nuevo médico había llegado al pueblo vecino de San Cristóbal, y que algunos habitantes habían comenzado a reportar fenómenos extraños similares a los que yo experimenté en mis primeros días en Umbraluna. "Doctor Mendoza", escribía con su caligrafía elegante, "creo que el joven doctor Sebastián Vega podría necesitar la guía que usted una vez necesitó. Los espíritus inquietos no se limitan a nuestro pequeño pueblo, y temo que él esté pasando por la misma confusión y terror que usted experimentó al principio." La carta me hizo reflexionar sobre la posibilidad de que lo que viví en Umbraluna no fuera un fenómeno aislado. Quizás hay lugares en este mundo donde el velo entre los vivos y los muertos es más delgado, donde ciertas personas con dones especiales son llamadas a servir como puentes entre ambos mundos. He decidido escribir al doctor Vega, ofreciéndole mi consejo y experiencia si la necesita. No puedo regresar físicamente a esa vida —mi alma aún está sanando de todo lo que experimenté—, pero puedo compartir lo que aprendí para que otros no tengan que enfrentar solos estos desafíos. También he comenzado a documentar sistemáticamente todos mis casos de Umbraluna en un tratado médico que espero publicar algún día. Lo he titulado "Sobre las Dolencias del Alma: Un Estudio de Fenómenos Inexplicables en la Práctica Médica Rural". Mi objetivo es presentar estas experiencias de manera que puedan ser útiles para otros médicos que enfrenten situaciones similares, sin sonar como un lunático. El mayor desafío es encontrar el lenguaje científico adecuado para describir experiencias que trascienden los límites de la ciencia actual. Estoy desarrollando una nueva terminología que combina conceptos médicos tradicionales con observaciones sobre fenómenos paranormales. Por ejemplo, he acuñado el término "dolor emocional residual" para describir el sufrimiento que algunos espíritus experimentan después de la muerte. También he creado clasificaciones para diferentes tipos de manifestaciones sobrenaturales basadas en su origen emocional: apariciones de culpa, espíritus de amor no correspondido, almas atrapadas por promesas incumplidas, etc. Mi esperanza es que, con el tiempo, la medicina evolucione para reconocer que la curación a veces requiere atender no solo al cuerpo físico, sino también a dimensiones más profundas de la experiencia humana. Mientras te relato todo esto, amigo mío, no puedo evitar notar cómo han cambiado mis perspectivas sobre la vida y la muerte. Antes de Umbraluna, veía la muerte como el final absoluto, como el momento en que toda conciencia cesaba. Ahora entiendo que es más bien una transición, y que para algunos, esa transición puede ser tan difícil como cualquier crisis que enfrentamos en vida. Esta comprensión ha transformado no solo mi práctica médica, sino también mi forma de vivir. Ahora presto más atención a resolver conflictos antes de que sea demasiado tarde, a expresar amor y perdón mientras aún hay tiempo, a vivir de manera que, cuando llegue mi hora, pueda partir en paz. Hace poco tuve una experiencia que me confirmó que mi conexión con el mundo espiritual, aunque más tenue que antes, aún persiste. Estaba atendiendo a una anciana en sus últimas horas cuando noté una presencia familiar en la habitación. Era el espíritu de su esposo, que había muerto años antes, esperando pacientemente para acompañarla en su transición. No dije nada a la familia —he aprendido que no todos están listos para escuchar estas cosas—, pero la imagen me llenó de una profunda paz. Me recordó que la muerte no siempre es una experiencia solitaria y aterradora. A veces es un reencuentro, una reconciliación, un regreso a casa. Esa noche, después de que la anciana partiera serenamente, encendí dos velas en lugar de una: una por su alma, y otra en agradecimiento por la revelación de que el amor verdadero trasciende incluso la muerte. Si hay una lección principal que quiero transmitirte de toda esta extraordinaria experiencia, es esta: mantén tu mente abierta a las posibilidades que van más allá de lo que puedes ver y tocar. La realidad es más vasta y misteriosa de lo que nuestra ciencia actual puede explicar. Pero también mantén tu compasión siempre despierta. Ya sea que trates con vivos o con muertos, con pacientes corporales o espíritus inquietos, lo que más necesitan es ser escuchados, comprendidos y tratados con dignidad. El mundo está lleno de sufrimiento, amigo mío, pero también está lleno de oportunidades para sanar, para consolar, para hacer la diferencia en la vida —y en la muerte— de otros. Mientras la noche se profundiza y el fuego de la chimenea se reduce a brasas, siento que he logrado transmitirte la esencia de lo que viví en Umbraluna. No sé si volveré alguna vez a ese pequeño pueblo entre las montañas, pero sé que una parte de mi alma permanecerá allí para siempre, cuidando a los vivos y velando por los muertos. Los susurros han cesado, pero la sabiduría que me trajeron permanece. Y ahora, querido amigo, esa sabiduría es también tuya. Que tengas sueños tranquilos, y que nunca olvides que en este vasto universo, todos estamos conectados por hilos invisibles de compasión y amor que ni siquiera la muerte puede romper. Dr. Aurelius Mendoza Capital del Reino Año 1850. '

In [14]:
print(len(gen_text_input))

19952


In [10]:
gen_text_input='Bienvenidos viajeros, soy El Carretero, y hoy los llevaré por senderos donde la realidad se desvanece y lo imposible cobra vida. Prepárense para adentrarse en historias que desafían la razón, donde las sombras susurran secretos ancestrales y fuerzas cósmicas despiertan desde las profundidades del tiempo. Suban a mi carreta... el viaje hacia lo desconocido está por comenzar.'

In [11]:
# chunk text into smaller pieces

def chunk_text(text, max_chars=135):
    """
    Splits the input text into chunks, each with a maximum number of characters.

    Args:
        text (str): The text to be split.
        max_chars (int): The maximum number of characters per chunk.

    Returns:
        List[str]: A list of text chunks.
    """
    chunks = []
    current_chunk = ""
    # Split the text into sentences based on punctuation followed by whitespace
    sentences = re.split(r"(?<=[;:,.!?])\s+|(?<=[；：，。！？])", text)
    # sentences = re.split(r'(?<=[;:.!?"])\s+|(?<=[；：。！？"])', text)
    # sentences = re.split(r'(?<=[;:.!?])\s+|(?<=[；：。！？])', text)
    for sentence in sentences:
        sentence=traducir_numero_a_texto(sentence)
        if len(current_chunk.encode("utf-8")) + len(sentence.encode("utf-8")) <= max_chars and (len(current_chunk.encode("utf-8")) == 0 or current_chunk[-2]== ","):
        # if len(current_chunk.encode("utf-8")) + len(sentence.encode("utf-8")) <= max_chars:
            current_chunk += sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence
            # print(f'{sentence}1{current_chunk}')
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())

            current_chunk = sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence
            # print(f'{sentence}2{current_chunk}')

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


In [12]:
# ruta="./F5TTS/Presentaciones/"
# ruta="./F5TTS/NochesSanDamian/"
# ruta="./F5TTS/SusurrosUmbraluna/"
ruta="./F5TTS/circosalmasperdidas/"

# ema_model = F5TTS_ema_model

# if not gen_text_input.startswith(" "):
# 	gen_text_input = " " + gen_text_input
# if not gen_text_input.endswith(". "):
# 	gen_text_input += ". "

# gen_text_input = gen_text_input.lower()
# gen_text_input = traducir_numero_a_texto(gen_text_input)

# print (gen_text_input)
# audio, sr = torchaudio.load(ref_audio)
# max_chars = int(len(ref_text.encode("utf-8")) / (audio.shape[-1] / sr) * (25 - audio.shape[-1] / sr))
# print(max_chars)
# gen_text_batches = chunk_text(gen_text_input, max_chars=max_chars)
original_batches = chunk_text(gen_text_input, max_chars=80)
# gen_text_batches = sentences_text(gen_text_input)

for batch in original_batches:
	# print(f"'{batch}',")
	print(f"'{batch}")


'Bienvenidos viajeros, soy El Carretero,
'y hoy los llevaré por senderos donde la realidad se desvanece y lo imposible cobra vida.
'Prepárense para adentrarse en historias que desafían la razón,
'donde las sombras susurran secretos ancestrales y fuerzas cósmicas despiertan desde las profundidades del tiempo.
'Suban a mi carreta...
'el viaje hacia lo desconocido está por comenzar.


In [14]:
def procesatexto(s):
	s=s.replace(" etc.","etcétera")
	s=s.replace("Dr. ","doctor")
	s=s.replace(";",",")
	s=s.replace(" XIX","19")
	# r=r.replace("¡","")
	# r=r.replace("!","")
	s=traducir_numero_a_texto(s).lower()
	return s

## Infiriendo Step by Step

In [15]:
# _ref_audio_cache = {}
# load asr pipeline
# asr_pipe = None
vocoder = load_vocoder()
# load models
F5TTS_model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
F5TTS_ema_model = load_model(
    DiT, F5TTS_model_cfg, "./F5TTS/model_1250000.safetensors"
    # DiT, F5TTS_model_cfg, "./F5TTS/model_1200000.safetensors"
)
model_obj = F5TTS_ema_model
audio, sr = torchaudio.load(ref_audio)
try:
	os.mkdir(ruta)
	print("RUTA CREADA",ruta)
except:
	 print("La ruta ya existe",ruta)

progress=tqdm

if audio.shape[0] > 1:
	audio = torch.mean(audio, dim=0, keepdim=True)

rms = torch.sqrt(torch.mean(torch.square(audio)))
if rms < target_rms:
	audio = audio * target_rms / rms
if sr != target_sample_rate:
	resampler = torchaudio.transforms.Resample(sr, target_sample_rate)
	audio = resampler(audio)
audio = audio.to(device)

generated_waves = []
spectrograms = []

if len(ref_text[-1].encode("utf-8")) == 1:
	ref_text = ref_text + " "

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  ./F5TTS/vocab.txt
tokenizer :  custom
model :  ./F5TTS/model_1250000.safetensors 

La ruta ya existe ./F5TTS/circosalmasperdidas/


In [180]:
# seed_everything(1258)

In [73]:
i=40
intentos=3
# j=1
# for gen_text in progress.tqdm(gen_text_batches):
for gen_text in progress.tqdm(original_batches):
	# Prepare the text
	gen_text=procesatexto(gen_text)
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for j in range(intentos):
		print(f"Iniciando Inferencia {i:04d}.{j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]
			generated_mel_spec = generated.permute(0, 2, 1)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			# with open(f'{ruta}{i:04d}.{j}.wav', 'w') as f:
			os.makedirs(f'{ruta}{j:02d}', exist_ok=True)
			sf.write(f'{ruta}{j:02d}/{i:04d}.{j}.wav', generated_wave,target_sample_rate)
			
	# i+=1
	i+=10
	

  0%|                                                                                                                                                                 | 0/357 [00:00<?, ?it/s]

Iniciando Inferencia 0040.0
Iniciando Inferencia 0040.1
Iniciando Inferencia 0040.2


  0%|▍                                                                                                                                                      | 1/357 [00:12<1:15:59, 12.81s/it]

Iniciando Inferencia 0050.0
Iniciando Inferencia 0050.1
Iniciando Inferencia 0050.2


  1%|▊                                                                                                                                                      | 2/357 [00:25<1:15:57, 12.84s/it]

...

 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏| 355/357 [1:15:05<00:25, 12.99s/it]

Iniciando Inferencia 3590.0
Iniciando Inferencia 3590.1
Iniciando Inferencia 3590.2


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 356/357 [1:15:16<00:12, 12.34s/it]

Iniciando Inferencia 3600.0
Iniciando Inferencia 3600.1
Iniciando Inferencia 3600.2


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 357/357 [1:15:27<00:00, 12.68s/it]


In [17]:
# gen_text_batches=[' , , Y los artistas... , ,']
# gen_text_batches=['—¡Bienvenidos, queridos niños!']
# gen_text_batches=['solo para verlos a ustedes.']
# gen_text_batches=['Esta noche tenemos un espectáculo que ningún niño debería perderse.']
# gen_text_batches=['El espectáculo está a punto de comenzar,']
# gen_text_batches=['Y ahora, damas y caballeros, el momento que todos han estado esperando!']
# gen_text_batches=['Hay uno que se nos escapa!']
# gen_text_batches=['Esta es una de ellas.']
# gen_text_batches=['podría cambiar la forma en que ves las sombras.']
gen_text_batches=['Algo extraño sucedió en el pueblo de Valdecima.']
# gen_text_batches=[' , solo para verlos a ustedes.']
i=14
j=0

intentos=3
# i+=5
for _, gen_text in enumerate(progress.tqdm(gen_text_batches)):
# Prepare the text
	gen_text=procesatexto(gen_text)
	print (gen_text, len(gen_text))
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for _ in range(intentos):
		print(f"Iniciando Inferencia {j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]

			#########################################################33
			if ref_audio_len >= generated.shape[1]:
				print(f"[WARN] ref_audio_len ({ref_audio_len}) >= generated length ({generated.shape[1]}), skipping slice.")
				generated_trimmed = generated  # o considera usar generated[:, -1:, :] como fallback
			else:
				generated_trimmed = generated[:, ref_audio_len:, :]

			generated_mel_spec = generated_trimmed.permute(0, 2, 1)

			###########################################################
			# print("generated shape:", generated.shape)
			# print("generated_mel_spec shape:", generated_mel_spec.shape)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			sf.write(f'{ruta}{i:04d}.{j}.wav', generated_wave,target_sample_rate)
		j+=1
	i+=1
	j=0

  0%|                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]

algo extraño sucedió en el pueblo de valdecima. 47
Iniciando Inferencia 0
[WARN] ref_audio_len (1359) >= generated length (267), skipping slice.
Iniciando Inferencia 1
[WARN] ref_audio_len (1359) >= generated length (267), skipping slice.
Iniciando Inferencia 2


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00, 14.53s/it]

[WARN] ref_audio_len (1359) >= generated length (267), skipping slice.


In [24]:
# Combine all generated waves with cross-fading
if cross_fade_duration <= 0:
	# Simply concatenate
	final_wave = np.concatenate(generated_waves)
else:
	final_wave = generated_waves[0]
	for i in range(1, len(generated_waves)):
		prev_wave = final_wave
		next_wave = generated_waves[i]

		# Calculate cross-fade samples, ensuring it does not exceed wave lengths
		cross_fade_samples = int(cross_fade_duration * target_sample_rate)
		cross_fade_samples = min(cross_fade_samples, len(prev_wave), len(next_wave))

		if cross_fade_samples <= 0:
			# No overlap possible, concatenate
			final_wave = np.concatenate([prev_wave, next_wave])
			continue

		# Overlapping parts
		prev_overlap = prev_wave[-cross_fade_samples:]
		next_overlap = next_wave[:cross_fade_samples]

		# Fade out and fade in
		fade_out = np.linspace(1, 0, cross_fade_samples)
		fade_in = np.linspace(0, 1, cross_fade_samples)

		# Cross-faded overlap
		cross_faded_overlap = prev_overlap * fade_out + next_overlap * fade_in

		# Combine
		new_wave = np.concatenate(
			[prev_wave[:-cross_fade_samples], cross_faded_overlap, next_wave[cross_fade_samples:]]
		)

		final_wave = new_wave

# Create a combined spectrogram
combined_spectrogram = np.concatenate(spectrograms, axis=1)
final_sample_rate=target_sample_rate
# return final_wave, target_sample_rate, combined_spectrogram

In [ ]:
# if remove_silence:
with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
	sf.write(f.name, final_wave, final_sample_rate)
	remove_silence_for_generated_wav(f.name)
	final_wave, _ = torchaudio.load(f.name)
final_wave = final_wave.squeeze().cpu().numpy()

In [ ]:
sf.write(ruta+'FuenteJuventudComplete.wav', final_wave,final_sample_rate)

ffmpeg -i entrada.wav -af "aecho=0.8:0.9:60:0.3" salida.wav
* in_gain=0.8: El sonido original mantiene el 80% de su volumen.
* out_gain=0.9: El eco es casi tan fuerte como la voz original.
* delay=60: El eco aparece 60 ms después.
* decay=0.3: El eco se atenúa un 30% respecto al original (eco más suave).
* highpass=f=1000 Se escucha como si estuviera por telefono
* lowpass=f=100 Como si estuvieras hablando con una almohada en la boca

In [ ]:
#Modificando Tonalidad
rr="./F5TTS/circosalmasperdidas/temp"

# !ffmpeg -y -i {rr}/1220.0.wav -af "aecho=0.8:0.88:60:0.4,asetrate=24000*0.85,atempo=1.20,highpass=f=200,lowpass=f=3000" {rr2}/1220.1.wav
# !ffmpeg -y -i {rr}/1220.0.wav -af "asetrate=24000*0.90,atempo=1.1,highpass=f=200,lowpass=f=3000" {rr2}/1220.1.wav
# !ffmpeg -y -i {rr}/1220.0.wav -af "asetrate=24000*0.90,atempo=1.2,highpass=f=200,lowpass=f=3000" {rr2}/1220.2.wav
# !ffmpeg -y -i {rr}/1220.0.wav -af "asetrate=24000*1.1,atempo=0.9,highpass=f=200,lowpass=f=3000" {rr2}/1220.3.wav
# !ffmpeg -y -i {rr}/1220.0.wav -af "asetrate=24000*0.95,atempo=1.05,highpass=f=200,lowpass=f=3000" {rr2}/1220.4.wav
# !ffmpeg -y -i {rr}/1220.0.wav -af "asetrate=24000*0.90,atempo=1.2,highpass=f=200,lowpass=f=3000" {rr2}/1220.5.wav

#Voz de robot
# !ffmpeg -y -i {rr}/1220.0.wav -af "aecho=0.6:0.8:40|80|120:0.5|0.4|0.3" {rr2}/1220.e.wav


# !ffmpeg -y -i {rr}/1220.2.wav -af "atempo=1.2" {rr2}/1220.2.r.wav
# !ffmpeg -y -i {rr}/1230.8.wav -af "atempo=1.2" {rr2}/1230.8.r.wav
# !ffmpeg -y -i {rr2}/1220.f.wav -af "asetrate=24000*1.1,atempo=0.9,highpass=f=200,lowpass=f=3000" {rr2}/1220.f2.wav


#PresentadorEco
# !ffmpeg -y -i {rr}/1900.0.wav -af "aecho=1:0.9:100|150:0.4|0.2,atempo=0.85,highpass=f=200,lowpass=f=3000" {rr}/1900.e5.wav
# !ffmpeg -y -i {rr}/1920.5.wav -af "aecho=1:0.9:100|150:0.4|0.2,atempo=0.85,highpass=f=200,lowpass=f=3000" {rr}/1920.e5.wav


#PresentadorEnojado Siseando la voz como metal siendo arrastrado sobre piedra
# !ffmpeg -y -i {rr}/2220.4.wav -af "asetrate=24000*0.95,atempo=1.1,treble=g=30,highpass=f=200,lowpass=f=3000" -ar 24000 {rr}/2220.0r.wav
# !ffmpeg -y -i {rr}/2240.0.wav -af "asetrate=24000*0.95,atempo=1.1,treble=g=40,highpass=f=200,lowpass=f=3000" -ar 24000 {rr}/2240.2r.wav




ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

In [65]:
import librosa
import soundfile as sf

rr="./F5TTS/circosalmasperdidas/temp"
# rr2="./F5TTS/circosalmasperdidas"

# n_steps indica cuántos semitonos subes o bajas:
# +12 = una octava arriba
# -12 = una octava abajo
# -3 = tono más profundo, efecto fantasmal sutil

# Cargar audio
y, sr = librosa.load(f"{rr}/2220.4.wav", sr=None)

# Cambiar pitch: -3 semitonos (baja) o +3 (sube)
# y_shifted = librosa.effects.pitch_shift(y, sr, n_steps=-3)
# Cambiar el pitch (por ejemplo, -3 semitonos)
y_shifted = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=-1)
# Guardar
sf.write(f"{rr}/2220.g1.wav", y_shifted, sr)

In [ ]:
# Re-importar librerías y redefinir rutas ya que el estado fue reiniciado
import librosa
import soundfile as sf
import numpy as np
from scipy.signal import fftconvolve

rr="./F5TTS/circosalmasperdidas"
rr2="./F5TTS/circosalmasperdidas"

# Ruta del archivo original re-subido
input_path = rr+"/1220.0.wav"
output_path = rr2+"/voz_amigable.wav"

# Cargar el audio original
y, sr = librosa.load(input_path, sr=None)

# 1. Subir el pitch ligeramente (+2 semitonos)
y_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=2)

# 2. Bajar un poco la velocidad (tempo más lento pero amigable)
y_stretch = librosa.effects.time_stretch(y_pitch, rate=0.95)

# 3. Agregar una reverb suave (impulso artificial simple)
impulse = np.zeros(int(sr * 0.1))
impulse[0] = 1.0
impulse[int(sr * 0.05)] = 0.4
impulse[int(sr * 0.1) - 1] = 0.2
y_reverb = fftconvolve(y_stretch, impulse, mode='full')[:len(y_stretch)]

# 4. Normalizar el volumen
y_final = y_reverb / np.max(np.abs(y_reverb))

# Guardar el resultado
sf.write(output_path, y_final, sr)

output_path

'./F5TTS/circosalmasperdidas/voz_amigable.wav'